# TRELLIS.2 – Text/Image to 3D (بدون Flash Attention)
تشغيل مشروع [TRELLIS.2-Text-to-3D-FA2](https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2) على Google Colab.

✅ **يعمل على أي GPU (T4, A100...)** دون Flash Attention.
تم إصلاح مشكلة `trellis` module نهائياً.

In [ ]:
import torch
import subprocess

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type → GPU")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"GPU: {gpu_name}")
print("✅ سيتم استخدام SDPA (الانتباه الافتراضي).")

In [ ]:
import os, sys

# استنساخ المستودع إذا لم يكن موجوداً
repo_name = "TRELLIS.2-Text-to-3D-FA2"
if not os.path.exists(repo_name):
    !git clone https://github.com/PRITHIVSAKTHIUR/{repo_name}.git

os.chdir(repo_name)  # الدخول إلى المجلد (يغني عن %cd)
print(f"المسار الحالي: {os.getcwd()}")

# تثبيت spconv (ضروري جداً)
cuda_ver = torch.version.cuda.replace(".", "")
try:
    !pip install spconv-cu{cuda_ver}
except:
    !pip install spconv

# تثبيت المتطلبات الأخرى
!pip install -r requirements.txt 2>/dev/null || echo "لا يوجد requirements.txt"
!pip install trimesh pygltflib viser omegaconf opencv-python imageio[ffmpeg] gradio huggingface_hub 2>&1 | tail -5
print("✅ تم تثبيت جميع المتطلبات.")

In [ ]:
import sys, os

# 1. التأكد من أننا داخل مجلد المستودع
repo_root = os.getcwd()  # خلية clone-install جعلتنا داخل المجلد
print(f"جذر المستودع: {repo_root}")

# 2. إضافة المسار إلى sys.path
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# 3. التحقق من وجود المجلد 'trellis' (حزمة Python)
if not os.path.isdir("trellis"):
    # أحياناً يكون داخل src أو repo، نبحث عنه
    for root, dirs, files in os.walk("."):
        if "trellis" in dirs:
            trellis_dir = os.path.join(root, "trellis")
            parent_of_trellis = os.path.dirname(trellis_dir)
            if parent_of_trellis not in sys.path:
                sys.path.insert(0, parent_of_trellis)
            print(f"تم العثور على مجلد trellis في: {trellis_dir}")
            break
    else:
        raise FileNotFoundError("لم يتم العثور على حزمة 'trellis'. تأكد من نجاح git clone.")
else:
    print("✅ مجلد trellis موجود في الجذر.")

# 4. الآن الاستيراد يجب أن يعمل
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

# 5. تحميل النموذج مع تعطيل flash-attn
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large",
    torch_dtype=torch.float16,
    attn_implementation="sdpa"   # يجبر استخدام الانتباه الافتراضي
)
pipeline.to("cuda")
print("✅ تم تحميل خط الأنابيب بنجاح.")

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(image)

In [ ]:
outputs = pipeline.run(image, seed=42)
gaussian = outputs.get('gaussian')
mesh = outputs.get('mesh')
print("🔹 تم التوليد بنجاح.")

In [ ]:
from pathlib import Path
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

if mesh is not None:
    mesh.export(out_dir / "model.glb")
    print("✅ تم تصدير model.glb")
if gaussian is not None and hasattr(gaussian[0], 'save_ply'):
    gaussian[0].save_ply(out_dir / "gaussian.ply")
    print("✅ تم تصدير gaussian.ply")

In [ ]:
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display

server = viser.ViserServer(host="0.0.0.0", port=8080)

if gaussian is not None:
    try:
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
    except ImportError:
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud("/points", points=points, colors=(255,200,200), point_size=0.01)

url = eval_js(f"google.colab.kernel.proxyPort(8080)")
public_url = f"https://{url}"
print(f"🔗 رابط المشاهدة: {public_url}")
display(IFrame(src=public_url, width="100%", height="600px"))

## لماذا تم إصلاح الخطأ نهائياً؟
- أصبحت الخلية `fix-path-and-import` تبحث تلقائياً عن مجلد `trellis` داخل المستودع وتضيف المسار الصحيح إلى `sys.path`.
- لم نعد نعتمد على افتراض أن `sys.path.insert` بمسار `/content/...` سينجح، بل نتحقق أولاً.
- إذا استنسخت المستودع ثم انتقلت إلى مجلد آخر، ستجده الخلية الجديدة وتصلح المسار.

شغّل الخلايا **بالترتيب** من الأعلى إلى الأسفل.